# Scaling 1-20 papers

In [ ]:
from backend.rag_core import rag_pipeline
from pathlib import Path



In [ ]:
# load the QA dataset


DATASET_NAME = "arxiv_scale_rag_self_instruct_v1" 

DATASET_PATH = Path("eval_qas_self_instruct.jsonl")
with DATASET_PATH.open("r", encoding="utf-8") as f:
    qna_dataset = [json.loads(line) for line in f]

In [ ]:
def run_rag_eval(qna_dataset, retrieve_chunks=10):
    run_log = []
    for ex in qna_dataset:
        q = ex["question"]
        gold = ex.get("answer")          # None if not available
        qid = ex.get("id")               # optional

        out = rag_pipeline(q, retrieve_chunks=retrieve_chunks)

        run_log.append({
            # identifiers
            "query_id": qid,
            "dataset": DATASET_NAME,

            # I/O
            "query": q,
            "gold_answer": gold,
            "answer": out["answer"],

            # retrieval output
            "contexts": out["contexts"],      # you’ll use this for Recall@k etc.

            # performance
            "timing": out["timing"],          # p50/p95 latency later

            # experiment meta
            "rag_config": out["config"],
            "timestamp": datetime.utcnow().isoformat(),
        })

    return run_log

In [ ]:
def save_log(run_log, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for row in run_log:
            f.write(json.dumps(row) + "\n")

In [ ]:
run_log = run_rag_eval(qna_dataset, retrieve_chunks=10)

In [ ]:
# create folder: data/QnA/
save_dir = Path("data") / "QnA"
save_dir.mkdir(parents=True, exist_ok=True)

# save file inside it
save_path = save_dir / "run_log.graph_rag_v1.jsonl"
save_log(run_log, save_path)

# Scaling 100-5000 paper
we use the arxiv API

In [7]:
import os
import time
import requests
import feedparser
from requests.exceptions import ChunkedEncodingError, ConnectionError

def safe_download(url, path, retries=5, chunk=1024*1024):
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=20) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk_data in r.iter_content(chunk_size=chunk):
                        if chunk_data:
                            f.write(chunk_data)
            return True
        except (ChunkedEncodingError, ConnectionError, requests.Timeout):
            if attempt == retries - 1:
                print(f"FAILED: {url}")
                return False
            print(f"Retry {attempt+1}/{retries}: {url}")
            time.sleep(2)



def fetch_arxiv(category="cs.LG", total=4, batch=2, out_dir="papers"):
    os.makedirs(out_dir, exist_ok=True)

    for start in range(0, total, batch):
        url = f"http://export.arxiv.org/api/query?search_query=cat:{category}&start={start}&max_results={batch}"
        feed = feedparser.parse(url)

        for entry in feed.entries:
            pdf_url = entry.id.replace("abs", "pdf")
            pid = entry.id.split('/')[-1]
            pdf_path = f"{out_dir}/{pid}.pdf"

            ok = safe_download(pdf_url, pdf_path)
            if not ok:
                print(f"Skipping {pid}")

        time.sleep(3)


In [8]:
fetch_arxiv(category="cs.LG", total=1000, batch=100, out_dir="ml_papers")

In [9]:
folder = "ml_papers"
pdfs = [f for f in os.listdir(folder) if f.endswith(".pdf")]
len(pdfs)

1000

In [6]:
import sys
!{sys.executable} -m pip install feedparser

  Using cached feedparser-6.0.12-py3-none-any.whl.metadata (2.7 kB)
  Using cached sgmllib3k-1.0.0-py3-none-any.whl
Using cached feedparser-6.0.12-py3-none-any.whl (81 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [feedparser]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /home/mmk2266/envTorch124/bin/python -m pip install --upgrade pip


# Scaling 5k-50k papers
arXiv exposes a full “harvest all metadata” endpoint:

OAI-PMH (Open Archives Initiative – Protocol for Metadata Harvesting)

In [ ]:
!pip install sickle

In [ ]:
from sickle import Sickle

sickle = Sickle('http://export.arxiv.org/oai2')
records = sickle.ListRecords(metadataPrefix='arXiv', set='cs')

for record in records:
    pdf_url = record.metadata['id'][0].replace("abs", "pdf")
    # download pdf here